# RocWeather (All-Weather 200 Epochs) — Colab 训练、TensorBoard 与断点续训
- 数据集压缩包保存在 `/content/drive/MyDrive/RocWeather/archives/`
- 训练配置保存在 `/content/drive/MyDrive/RocWeather/configs/`（也可直接使用仓库 `configs/`）
- 每轮断点（`last.pt` / `best.pt` / `metrics.jsonl`）与 TensorBoard 日志实时写入 `/content/drive/MyDrive/RocWeather/runs/`

In [ ]:
# 1. 挂载 Google Drive、拉取代码、解压本地高速数据并启动内嵌 TensorBoard
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
from pathlib import Path

ARCHIVES = Path('/content/drive/MyDrive/RocWeather/archives')
if not Path('/content/RocWeather').is_dir():
    subprocess.run('git clone https://github.com/CHENHUI-X/RocWeather.git /content/RocWeather', shell=True, check=True)
subprocess.run('pip install -q -e "/content/RocWeather[tensorboard]"', shell=True, check=True)

Path('/content/data/train').mkdir(parents=True, exist_ok=True)
Path('/content/data/test').mkdir(parents=True, exist_ok=True)
if not (Path('/content/data/train/input').is_dir() and len(os.listdir('/content/data/train/input')) >= 18000):
    subprocess.run(f'unzip -o -q "{ARCHIVES}/allweather_train/allweather.zip" -d /content/data/train', shell=True, check=True)
shutil.copy2(ARCHIVES / 'train_manifest.txt', '/content/data/train/train.txt')

if not (Path('/content/data/test/input').is_dir() and len(os.listdir('/content/data/test/input')) >= 58):
    rd_tmp = Path('/content/rd_tmp')
    if rd_tmp.exists(): shutil.rmtree(rd_tmp)
    subprocess.run(f'unzip -o -q "{ARCHIVES}/raindrop/test_a.zip" -d "{rd_tmp}"', shell=True, check=True)
    for sub, src in [('input', rd_tmp / 'test_a' / 'data'), ('gt', rd_tmp / 'test_a' / 'gt')]:
        dst = Path('/content/data/test') / sub
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
shutil.copy2(ARCHIVES / 'test_raindropa_manifest.txt', '/content/data/test/test.txt')

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/RocWeather/runs --port 6006


In [ ]:
# 2. 启动 / 自动断点续训 200 轮（L4: BS=32 + BF16；T4 请切换为 adaptive_allweather_t4.yaml + fp16）
import shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/RocWeather')
CFG = DRIVE_ROOT / 'configs/adaptive_allweather_200ep_l4_bs32.yaml'
RUN_DIR = DRIVE_ROOT / 'runs/adaptive_allweather_200ep_l4_bs32'
LAST = RUN_DIR / 'last.pt'

if LAST.is_file():
    print('检测到断点，自动续训至 200 轮...')
    cmd = f"cd /content/RocWeather && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONUNBUFFERED=1 python3 -m rocweather train --resume '{LAST}' --epochs 200 --device cuda --precision bf16"
else:
    if RUN_DIR.exists(): shutil.rmtree(RUN_DIR)
    print('开始全新 200 轮训练...')
    cmd = f"cd /content/RocWeather && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONUNBUFFERED=1 python3 -m rocweather train --config '{CFG}' --output '{RUN_DIR}' --device cuda --precision bf16"

!{cmd}
